# Challenge 02 — News Agent (Code-First)

In this challenge, you'll build a **News Agent** programmatically using the **Azure AI Agent SDK** and Python. By the end, you'll be able to chat with your agent from this notebook.

## Learning objectives
- Authenticate to Azure AI Foundry from code
- Create an agent with custom instructions
- Send messages and process agent runs
- Inspect agent responses

Set up your Python environment for the notebook

In [ ]:
# From inside the WhatTheHack folder
python -m venv .venv

# Activate it 
source .venv/bin/activate


# Upgrade pip
python -m pip install --upgrade pip

# Install the core packages you'll need for the News Agent challenge
pip install azure-ai-projects azure-ai-agents azure-identity ipykernel python-dotenv requests

 Imports & env setup

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import MessageRole

# Load env vars
load_dotenv()
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")  # optional but recommended

assert PROJECT_ENDPOINT, "PROJECT_ENDPOINT is missing from .env"
assert MODEL_DEPLOYMENT_NAME, "MODEL_DEPLOYMENT_NAME is missing from .env"
assert "/api/projects/" in PROJECT_ENDPOINT, (
    "PROJECT_ENDPOINT must be a Foundry *project* endpoint, e.g. "
    "https://<resource>.services.ai.azure.com/api/projects/<project-name> "
    f"(got: {PROJECT_ENDPOINT})"
)

print(f"Endpoint: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL_DEPLOYMENT_NAME}")
print(f"Tenant: {TENANT_ID or '(not set — will use CLI default)'}")

Cell 2 — Create the AgentsClient


In [ ]:
# Connect to your Foundry project.
# Try Azure CLI first; fall back to a browser sign-in if the CLI session isn't visible to the kernel.
credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
print("✅ Connected to Azure AI Foundry")

Create the News Agent

In [ ]:
# Define the agent's personality + job
news_agent = agents_client.create_agent(
    model=MODEL_DEPLOYMENT_NAME,
    name="news-travel-agent",
    instructions=(
        "You are a helpful travel news assistant. "
        "When a user asks about a destination, summarize the latest news, "
        "events, and travel advisories for that location in a friendly, concise way. "
        "Always end with a fun tip about visiting that place."
    )
)
print(f"✅ Agent created — ID: {news_agent.id}")

Create a thread + send a message


In [ ]:
# Start a conversation thread
thread = agents_client.threads.create()
print(f"✅ Thread created — ID: {thread.id}")

# Send a user message
message = agents_client.messages.create(
    thread_id=thread.id,
    role=MessageRole.USER,
    content="What's happening in Texas this week that a traveler should know?"
)
print(f"✅ Message sent")

Run the agent + see the response

In [ ]:
# Run the agent and wait for it to finish
run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=news_agent.id
)
print(f"Run status: {run.status}")

# Print the agent's reply
messages = agents_client.messages.list(thread_id=thread.id)
for msg in messages:
    role = msg.role
    text = msg.content[0].text.value if msg.content else ""
    print(f"\n--- {role.upper()} ---\n{text}")

 Cleanup


In [ ]:
# Delete the agent and thread to keep your Foundry project clean
agents_client.delete_agent(news_agent.id)
print(f"Agent deleted - ID: {news_agent.id}")

agents_client.threads.delete(thread.id)
print(f"Thread deleted - ID: {thread.id}")